# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [21]:
from openai import OpenAI
import os

from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')


In [22]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [23]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['employees', 'salary', 'studies'],
        'definition': ['Employee information, name...', 'Salary details for each year', 'Educational studies, name of the institution, type of studies, level']}
df = pd.DataFrame(data)
print(df)

       table                                         definition
0  employees                      Employee information, name...
1     salary                       Salary details for each year
2    studies  Educational studies, name of the institution, ...


In [24]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [25]:
print(text_tables)

employees: Employee information, name...
salary: Salary details for each year
studies: Educational studies, name of the institution, type of studies, level


In [26]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [27]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question='what is the average salary for the people with Master?')

In [28]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees", "salary", "studies"]
}
```


In [30]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question='what is the average salary for the people with Master?')

In [31]:
print(return_OAI(pqt3))

```json
{
    "tables": ["employees", "salary", "studies"]
}
```


In [33]:
# Define a clearer and more structured prompt template
prompt_question_tables = """
You are an expert database assistant.
Given the following SQL tables and their schema:

### Tables
{tables}

Determine which tables are required to answer the user's question.
Return the result as a valid JSON list of table names.

### User Question
{question}
"""

# Example: user question and table definitions
user_questions = ["List all employees who joined after 2020.",
    "What is the total revenue by department?"
]

# Generate and query the model for each question
for q in user_questions:
    formatted_prompt = prompt_question_tables.format(tables=text_tables, question=q)
    print(f"🔹 User Question: {q}\n")
    print(return_OAI(formatted_prompt))
    print("-" * 60)


🔹 User Question: List all employees who joined after 2020.

To answer the user's question, the following tables are required:

```json
["employees"]
```
------------------------------------------------------------
🔹 User Question: What is the total revenue by department?

```json
["employees", "salary"]
```
------------------------------------------------------------


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?